# 25 — OpenRouter ReAct Agent (from scratch)

**Pattern:** Manual ReAct loop — reason → act → observe — with only the `openai` SDK.  
**Key insight:** Orchestration frameworks collapse this `while` loop into one function call.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/25-openrouter-react-agent/openrouter_react_agent_workbook.ipynb)

In [ ]:
%pip install -q openai pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

## 1. What is a ReAct loop?

ReAct = **Re**ason + **Act**. The agent:

1. **Reasons** -- calls the model; it decides whether to use a tool or answer directly
2. **Acts** -- if the model calls a tool, we execute it
3. **Observes** -- we append the tool result to the message history
4. **Repeats** -- until the model returns a plain answer (no tool calls)

LangGraph's `create_react_agent()` does exactly this -- this workbook shows the raw version.

## 2. Tool definitions

In [ ]:
import json
from openai import OpenAI

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                "required": ["a", "b"],
            },
        },
    },
]

DISPATCH = {
    "add": lambda a, b: a + b,
    "multiply": lambda a, b: a * b,
}

## 3. The ReAct loop

In [ ]:
def react(question: str, model: str = "openai/gpt-4o-mini") -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    messages = [
        {"role": "system", "content": "You are a math assistant. Use the provided tools. Do not compute yourself."},
        {"role": "user", "content": question},
    ]
    step = 0
    while True:
        step += 1
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, tool_choice="auto"
        )
        msg = response.choices[0].message

        if not msg.tool_calls:          # no tool call = final answer
            return msg.content

        messages.append(msg.model_dump(exclude_unset=True))

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = DISPATCH[call.function.name](**args)
            print(f"  [step {step}] {call.function.name}({args}) = {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

## 4. Run

In [ ]:
for q in ["What is (3 + 4) * 5?", "What is 100 + (6 * 7)?"]:
    print(f"Q: {q}")
    print(f"A: {react(q)}\n")

## 5. Starter Exercise

Add a `subtract` tool and test with `"What is 50 - (3 * 4)?"`

In [ ]:
# Your code here

### Answer Key

In [ ]:
TOOLS_V2 = TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "subtract",
            "description": "Subtract b from a.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                "required": ["a", "b"],
            },
        },
    }
]
DISPATCH_V2 = {**DISPATCH, "subtract": lambda a, b: a - b}


def react_v2(question: str, model: str = "openai/gpt-4o-mini") -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    messages = [
        {"role": "system", "content": "You are a math assistant. Use the provided tools."},
        {"role": "user", "content": question},
    ]
    step = 0
    while True:
        step += 1
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS_V2, tool_choice="auto"
        )
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_unset=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = DISPATCH_V2[call.function.name](**args)
            print(f"  [step {step}] {call.function.name}({args}) = {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})


print(react_v2("What is 50 - (3 * 4)?"))